# Sesión 11 — Modelos Ocultos de Markov (HMM)
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo III · Modelos Generativos Clásicos**

## Objetivos de aprendizaje

Al finalizar esta sesión serás capaz de:

1. Definir los tres componentes de un HMM (π, A, B) y generar secuencias sintéticas.
2. Implementar el **algoritmo Forward** en log-espacio para calcular P(X|λ).
3. Implementar el **algoritmo Backward** y calcular las probabilidades posteriores suavizadas.
4. Implementar el **algoritmo de Viterbi** para decodificar la secuencia de estados más probable.
5. Aplicar **Baum-Welch** (EM para HMM) via hmmlearn para estimar parámetros a partir de datos.
6. Aplicar el HMM a la estadificación del sueño de 5 estados (W/N1/N2/N3/REM) y a la segmentación de microestados de EEG.

## Aplicación principal

**Estadificación del sueño** — 5 estados según el manual AASM 2015:
Vigilia (W), Sueño ligero (N1), Sueño intermedio (N2), Sueño profundo (N3), REM.

> Berry, R.B. et al. (2015). *AASM Scoring Manual* (versión 2.2). American Academy of Sleep Medicine.
> https://aasm.org/clinical-resources/scoring-manual/

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Rabiner, L.R. (1989). A tutorial on hidden Markov models and selected applications in speech recognition. *Proc. IEEE*, 77(2), 257–286. |
| ★★★ | Bishop, C.M. (2006). *PRML*. §13.1–13.2 (HMM). Springer. |
| ★★☆ | Längkvist, M., Karlsson, L. & Loutfi, A. (2012). Sleep stage classification using unsupervised feature learning. *Advances in AI*, 2012, 107046. https://doi.org/10.1155/2012/107046 |
| ★★☆ | Iber, C. et al. (2007). *The AASM Manual for the Scoring of Sleep and Associated Events*. American Academy of Sleep Medicine. |
| ★☆☆ | Perslev, M. et al. (2021). U-Sleep: resilient high-frequency sleep staging. *npj Digital Medicine*, 4, 72. https://doi.org/10.1038/s41746-021-00440-5 |

## Parte 0 — Configuración y parámetros del HMM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from scipy.special import logsumexp
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})

# ── Parámetros del HMM de estadificación del sueño ───────────────────────────
# Estados: 0=Vigilia, 1=N1 (ligero), 2=N2 (intermedio), 3=N3 (profundo), 4=REM
# Basado en: Berry, R.B. et al. (2015). AASM Scoring Manual v2.2.
# https://aasm.org/clinical-resources/scoring-manual/

nombres_estado = ['Vigilia', 'N1', 'N2', 'N3', 'REM']
colores_estado = ['#E8D44D', '#88C999', '#5B9BD5', '#2E4FA3', '#D45B5B']
K = 5  # número de estados

# Matriz de transición A[i,j] = P(z_{t+1}=j | z_t=i)
A_true = np.array([
    #  W     N1    N2    N3    REM
    [0.70, 0.20, 0.07, 0.01, 0.02],  # desde Vigilia
    [0.10, 0.50, 0.35, 0.04, 0.01],  # desde N1
    [0.02, 0.05, 0.70, 0.18, 0.05],  # desde N2
    [0.01, 0.02, 0.25, 0.68, 0.04],  # desde N3
    [0.08, 0.10, 0.12, 0.01, 0.69],  # desde REM
])
assert np.allclose(A_true.sum(axis=1), 1.0), 'Las filas de A deben sumar 1'

# Distribución inicial de estados
pi_true = np.array([0.60, 0.20, 0.10, 0.05, 0.05])

# Modelo de emisión gaussiano diagonal
# Características observadas: [Potencia delta, Potencia theta, Potencia alfa, Amplitud EMG]
obs_means = np.array([
    # Delta  Theta  Alfa   EMG
    [ 5.0,   8.0,  20.0,  8.0],  # Vigilia: alfa/EMG altos
    [12.0,  15.0,  10.0,  3.0],  # N1: theta dominante
    [25.0,   8.0,   5.0,  2.0],  # N2: delta creciente
    [45.0,   4.0,   2.0,  1.0],  # N3: delta dominante (ondas lentas)
    [ 8.0,  18.0,   8.0,  4.0],  # REM: theta, EMG bajo
])
obs_stds = np.array([
    [2.0, 3.0, 5.0, 3.0],
    [4.0, 5.0, 4.0, 1.5],
    [7.0, 3.0, 2.0, 0.8],
    [8.0, 2.0, 1.0, 0.5],
    [3.0, 6.0, 3.5, 2.0],
])

print('Parámetros del HMM definidos.')
print(f'Estados: {nombres_estado}')
print(f'Características: [Delta, Theta, Alfa, EMG]')
print(f'\nMatriz de transición A:')
print(np.round(A_true, 2))

## Parte 1 — Generación de secuencias y los tres problemas del HMM

Un HMM queda completamente especificado por la tripleta λ = (π, A, B):

| Componente | Símbolo | Dimensiones | Significado |
|---|---|---|---|
| Distribución inicial | π | (K,) | P(z₁ = k) |
| Matriz de transición | A | (K×K) | P(zₜ₊₁ = j \| zₜ = i) |
| Modelo de emisión | B | parámetros | P(xₜ \| zₜ = k) |

Los **tres problemas canónicos** del HMM (Rabiner, 1989):
1. **Evaluación:** ¿cuál es P(X|λ)? → Forward
2. **Decodificación:** ¿cuál es la secuencia z más probable? → Viterbi
3. **Aprendizaje:** ¿cómo estimar λ* que maximiza P(X|λ)? → Baum-Welch

In [ ]:
def sample_hmm(A, pi, obs_means, obs_stds, T, seed=None):
    """Genera T observaciones desde un HMM gaussiano diagonal."""
    rng_loc = np.random.default_rng(seed)
    K, d    = obs_means.shape
    estados = np.zeros(T, dtype=int)
    obs     = np.zeros((T, d))

    estados[0] = rng_loc.choice(K, p=pi)
    obs[0]     = rng_loc.normal(obs_means[estados[0]], obs_stds[estados[0]])

    for t in range(1, T):
        estados[t] = rng_loc.choice(K, p=A[estados[t-1]])
        obs[t]     = rng_loc.normal(obs_means[estados[t]], obs_stds[estados[t]])

    return estados, obs


# Generar una noche de sueño simulada: ~480 épocas de 30 s = 4 horas
T = 480
z_seq, X_seq = sample_hmm(A_true, pi_true, obs_means, obs_stds, T, seed=42)

print(f'Secuencia de {T} épocas generada.')
print(f'Distribución de estados: {dict(zip(nombres_estado, np.bincount(z_seq, minlength=K)))}')

# ── Visualizar el hipnograma y las características observadas ─────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

# Hipnograma
for t in range(T-1):
    axes[0].fill_between([t, t+1], [0, 0], [1, 1],
                          color=colores_estado[z_seq[t]], alpha=0.9, linewidth=0)
parches = [mpatches.Patch(color=c, label=n)
           for c, n in zip(colores_estado, nombres_estado)]
axes[0].legend(handles=parches, loc='upper right', fontsize=8, ncol=5)
axes[0].set(yticks=[], title='Hipnograma simulado (noche completa)', ylabel='Estado')

# Delta y Theta
axes[1].plot(X_seq[:, 0], color='#2E4FA3', lw=0.8, label='Delta (0.5–4 Hz)')
axes[1].plot(X_seq[:, 1], color='#88C999', lw=0.8, label='Theta (4–8 Hz)', alpha=0.7)
axes[1].set(ylabel='Potencia (μV²/Hz)', title='Características EEG observadas')
axes[1].legend(fontsize=8)

# Alfa y EMG
axes[2].plot(X_seq[:, 2], color='#5B9BD5', lw=0.8, label='Alfa (8–13 Hz)')
axes[2].plot(X_seq[:, 3], color='#D45B5B', lw=0.8, label='EMG', alpha=0.7)
axes[2].set(xlabel='Época (×30 s)', ylabel='Amplitud',
            title='Alfa y EMG — discriminan vigilia y REM')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Parte 2 — Algoritmo Forward: P(X|λ)

El algoritmo Forward calcula la verosimilitud marginal P(X|λ) eficientemente
mediante programación dinámica. Define:

$$\alpha_t(k) = P(x_1, \ldots, x_t, z_t = k \mid \lambda)$$

**Inicialización:**
$\alpha_1(k) = \pi_k \cdot b_k(x_1)$

**Recursión:**
$\alpha_{t+1}(j) = b_j(x_{t+1}) \sum_k \alpha_t(k) \cdot A_{kj}$

**Terminación:**
$P(\mathbf{X} \mid \lambda) = \sum_k \alpha_T(k)$

Todo en **log-espacio** con `logsumexp` para evitar underflow numérico.

In [ ]:
def log_emision(x, medias, stds):
    """Log p(xₜ | zₜ=k) para todos los k — gaussiana diagonal."""
    # x: (d,), medias: (K,d), stds: (K,d)
    log_probs = -0.5 * np.sum(
        ((x - medias) / stds)**2 + 2*np.log(stds) + np.log(2*np.pi),
        axis=1
    )
    return log_probs   # (K,)


def forward(X, A, pi, obs_means, obs_stds):
    """Algoritmo Forward en log-espacio. Retorna (log_alpha, log_verosimilitud)."""
    T, d   = X.shape
    K      = A.shape[0]
    log_A  = np.log(A + 1e-300)
    log_alpha = np.zeros((T, K))

    # Inicialización
    log_alpha[0] = np.log(pi + 1e-300) + log_emision(X[0], obs_means, obs_stds)

    # Recursión
    for t in range(1, T):
        log_b = log_emision(X[t], obs_means, obs_stds)
        for j in range(K):
            log_alpha[t, j] = log_b[j] + logsumexp(log_alpha[t-1] + log_A[:, j])

    log_lik = logsumexp(log_alpha[-1])
    return log_alpha, log_lik


def backward(X, A, obs_means, obs_stds):
    """Algoritmo Backward en log-espacio."""
    T, d  = X.shape
    K     = A.shape[0]
    log_A = np.log(A + 1e-300)
    log_beta = np.zeros((T, K))  # Inicialización: β_T(k) = 1 → log β_T(k) = 0

    for t in range(T-2, -1, -1):
        log_b = log_emision(X[t+1], obs_means, obs_stds)
        for k in range(K):
            log_beta[t, k] = logsumexp(log_A[k, :] + log_b + log_beta[t+1])

    return log_beta


# ── Calcular forward y backward ───────────────────────────────────────────────
log_alpha, log_lik = forward(X_seq, A_true, pi_true, obs_means, obs_stds)
print(f'Log P(X|λ_true) = {log_lik:.2f}  (por época: {log_lik/T:.3f})')

log_beta = backward(X_seq, A_true, obs_means, obs_stds)

# Probabilidades posteriores suavizadas γₜ(k) = P(zₜ=k | X, λ)
log_gamma  = log_alpha + log_beta
log_gamma -= logsumexp(log_gamma, axis=1, keepdims=True)
gamma      = np.exp(log_gamma)   # (T, K)

# ── Visualización ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Hipnograma verdadero
for t in range(T-1):
    axes[0].fill_between([t, t+1], [0, 0], [1, 1],
                          color=colores_estado[z_seq[t]], alpha=0.9, linewidth=0)
axes[0].legend(handles=parches, loc='upper right', fontsize=8, ncol=5)
axes[0].set(yticks=[], title='Secuencia verdadera de estados (hipnograma)')

# Probabilidades posteriores
im = axes[1].imshow(gamma.T, aspect='auto', cmap='hot_r', vmin=0, vmax=1,
                     extent=[0, T, K-0.5, -0.5])
axes[1].set_yticks(range(K))
axes[1].set_yticklabels(nombres_estado, fontsize=9)
axes[1].set(xlabel='Época', title='Posteriores suavizadas P(zₜ=k | X, λ) — Algoritmo Forward-Backward')
plt.colorbar(im, ax=axes[1], label='Probabilidad posterior')

plt.tight_layout()
plt.show()

# Exactitud del clasificador posterior (argmax)
z_posterior = gamma.argmax(axis=1)
acc_post    = (z_posterior == z_seq).mean()
print(f'Exactitud (argmax de posterior): {acc_post:.3f}')

## Parte 3 — Algoritmo de Viterbi: decodificación óptima

La posterior suavizada maximiza P(zₜ=k|X) en cada instante **por separado**.
El algoritmo de Viterbi encuentra la **secuencia completa** más probable:

$$\hat{\mathbf{z}} = \arg\max_{z_1,\ldots,z_T} P(z_1,\ldots,z_T \mid \mathbf{X}, \lambda)$$

Define: $\delta_t(k) = \max_{z_1,\ldots,z_{t-1}} \log P(x_1,\ldots,x_t, z_t=k, z_1,\ldots,z_{t-1} \mid \lambda)$

**Recursión:** $\delta_{t+1}(j) = \log b_j(x_{t+1}) + \max_k[\delta_t(k) + \log A_{kj}]$

La diferencia con la posterior: Viterbi garantiza que la secuencia decodificada
sea **globalmente consistente** (respeta las transiciones permitidas).

In [ ]:
def viterbi(X, A, pi, obs_means, obs_stds):
    """Algoritmo de Viterbi — retorna la secuencia de estados más probable."""
    T, d  = X.shape
    K     = A.shape[0]
    log_A = np.log(A + 1e-300)

    delta = np.full((T, K), -np.inf)
    psi   = np.zeros((T, K), dtype=int)

    # Inicialización
    delta[0] = np.log(pi + 1e-300) + log_emision(X[0], obs_means, obs_stds)

    # Recursión
    for t in range(1, T):
        log_b = log_emision(X[t], obs_means, obs_stds)
        for j in range(K):
            scores     = delta[t-1] + log_A[:, j]
            psi[t, j]  = np.argmax(scores)
            delta[t, j] = scores[psi[t, j]] + log_b[j]

    # Retroceso (backtracking)
    z_hat     = np.zeros(T, dtype=int)
    z_hat[-1] = np.argmax(delta[-1])
    for t in range(T-2, -1, -1):
        z_hat[t] = psi[t+1, z_hat[t+1]]

    return z_hat


z_viterbi = viterbi(X_seq, A_true, pi_true, obs_means, obs_stds)
acc_vit   = (z_viterbi == z_seq).mean()
print(f'Exactitud Viterbi (parámetros verdaderos): {acc_vit:.3f}')

# ── Comparación visual ────────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)

for ax, (z, titulo) in zip(axes[:2], [
    (z_seq,     'Hipnograma verdadero'),
    (z_viterbi, f'Viterbi decodificado  (exactitud={acc_vit:.2f})'),
]):
    for t in range(T-1):
        ax.fill_between([t, t+1], [0, 0], [1, 1],
                         color=colores_estado[z[t]], alpha=0.9, linewidth=0)
    ax.set(yticks=[], title=titulo)

# Errores
errores = (z_viterbi != z_seq).astype(float)
axes[2].fill_between(range(T), errores, alpha=0.7, color='tomato', step='mid')
axes[2].set(xlabel='Época', ylabel='Error', yticks=[0, 1],
            yticklabels=['correcto', 'incorrecto'],
            title='Errores de decodificación Viterbi')

axes[0].legend(handles=parches, loc='upper right', fontsize=8, ncol=5)
plt.tight_layout()
plt.show()

# Matriz de confusión
cm  = confusion_matrix(z_seq, z_viterbi)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=nombres_estado).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de confusión — Viterbi\n(parámetros verdaderos)')
plt.tight_layout()
plt.show()

print(f'\nExactitud Viterbi: {acc_vit:.3f}')
print(f'Exactitud argmax posterior: {acc_post:.3f}')
print('Viterbi es globalmente consistente; la posterior puede violar las transiciones.')

## Parte 4 — Baum-Welch: aprendizaje de parámetros HMM

El algoritmo **Baum-Welch** es el algoritmo EM aplicado a los HMMs:

**Paso E:** calcular
$\gamma_t(k) = P(z_t=k | X, \lambda)$ y
$\xi_t(k,j) = P(z_t=k, z_{t+1}=j | X, \lambda)$

**Paso M:** actualizar:
$\hat{A}_{kj} = \frac{\sum_t \xi_t(k,j)}{\sum_t \gamma_t(k)}$ ,
$\hat{\mu}_k = \frac{\sum_t \gamma_t(k) x_t}{\sum_t \gamma_t(k)}$

Se usa **hmmlearn** para la implementación eficiente, con un fallback
que ilustra la convergencia cuando la librería no está disponible.

In [ ]:
try:
    from hmmlearn import hmm as hmmlearn_hmm
    HAS_HMMLEARN = True
except ImportError:
    HAS_HMMLEARN = False
    print('hmmlearn no instalado. Instalar con: pip install hmmlearn --break-system-packages')

# ── Generar datos de entrenamiento: 10 noches ─────────────────────────────────
n_noches   = 10
secuencias = []
longitudes = []

for noche in range(n_noches):
    T_noche = int(rng.integers(420, 540))  # duración variable
    _, X_noche = sample_hmm(A_true, pi_true, obs_means, obs_stds,
                             T_noche, seed=noche)
    secuencias.append(X_noche)
    longitudes.append(T_noche)

X_todo = np.vstack(secuencias)
print(f'Datos de entrenamiento: {n_noches} noches, {X_todo.shape[0]} épocas totales')

if HAS_HMMLEARN:
    # Ajustar HMM con Baum-Welch
    modelo_bw = hmmlearn_hmm.GaussianHMM(
        n_components=K, covariance_type='diag',
        n_iter=100, tol=1e-4,
        verbose=False, random_state=0
    )
    modelo_bw.fit(X_todo, longitudes)

    print(f'\nBaum-Welch convergió: {modelo_bw.monitor_.converged}')
    print(f'Log-verosimilitud (train): {modelo_bw.score(X_todo, longitudes):.3f} por muestra')

    # Decodificar una noche de prueba
    # FIX: llama sample_hmm directamente — sin __wrapped__
    z_prueba_true, X_prueba = sample_hmm(A_true, pi_true, obs_means, obs_stds,
                                          480, seed=999)
    z_bw = modelo_bw.predict(X_prueba)

    # Visualizar matrices de transición: verdadera vs aprendida
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, (mat, titulo) in zip(axes, [
        (A_true,              'Matriz A verdadera'),
        (modelo_bw.transmat_, 'Matriz A aprendida (Baum-Welch)'),
    ]):
        im = ax.imshow(mat, cmap='Blues', vmin=0, vmax=1)
        ax.set(title=titulo,
               xticks=range(K), yticks=range(K),
               xticklabels=nombres_estado, yticklabels=nombres_estado)
        plt.colorbar(im, ax=ax)
        for i in range(K):
            for j in range(K):
                v = mat[i, j]
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                         fontsize=8, color='white' if v > 0.5 else 'black')

    plt.suptitle('Recuperación de parámetros — Baum-Welch\n'
                  '(el orden de estados puede diferir)', y=1.01)
    plt.tight_layout()
    plt.show()

else:
    # Ilustración de la convergencia de Baum-Welch (simulada)
    print('\nSimulando curva de convergencia de Baum-Welch...')
    n_iter  = 50
    ll_rand = -8.5
    ll_true = -5.2
    ll_curva = ll_rand + (ll_true - ll_rand) * (1 - np.exp(-0.12 * np.arange(n_iter)))
    ll_curva += rng.normal(0, 0.05, n_iter)
    ll_curva  = np.maximum.accumulate(ll_curva)  # monótona (propiedad EM)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(ll_curva, 'steelblue', lw=2.5)
    ax.axhline(ll_true, color='tomato', ls='--', lw=1.5,
                label=f'Log-lik modelo verdadero ({ll_true})')
    ax.set(xlabel='Iteración Baum-Welch', ylabel='Log-verosimilitud',
           title='Convergencia de Baum-Welch (simulada)\nEl EM nunca decrece la log-verosimilitud')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

## Parte 5 — Aplicación: segmentación de microestados de EEG

Los **microestados de EEG** son patrones topográficos de potencial eléctrico
que se mantienen casi estacionarios durante ~80–120 ms antes de transicionar
abruptamente. Cuatro configuraciones canónicas (A, B, C, D) se han identificado
en sujetos sanos (Lehmann et al., 1998).

El HMM captura naturalmente la **dinámica secuencial** de los microestados:
alta auto-transición + transiciones preferenciales entre clases.

In [ ]:
# ── Modelo de microestados EEG ────────────────────────────────────────────────
# 4 microestados canónicos (A, B, C, D), 250 Hz, épocas de 1 muestra
# Características: [GFP, asimetría lateral, índice antero-posterior]

nombres_ms = ['A', 'B', 'C', 'D']
colores_ms = ['steelblue', 'darkorange', 'seagreen', 'tomato']

# Matriz de transición: alta auto-transición (~80 ms a 250 Hz → ~20 muestras)
A_ms = np.array([
    [0.70, 0.10, 0.12, 0.08],
    [0.08, 0.72, 0.12, 0.08],
    [0.10, 0.10, 0.68, 0.12],
    [0.09, 0.09, 0.10, 0.72],
])
pi_ms = np.full(4, 0.25)

# Medias de observación: [GFP, asimetría lateral, índice antero-posterior]
ms_means = np.array([
    [3.2,  0.8,  0.1],   # A: predominio posterior derecho
    [3.5, -0.9,  0.2],   # B: predominio posterior izquierdo
    [4.0,  0.1, -0.8],   # C: predominio anterior
    [3.8,  0.1,  0.9],   # D: predominio central
])
ms_stds = np.full((4, 3), 0.6)

# Generar 8 segundos a 250 Hz = 2000 muestras
T_ms = 2000
z_ms, X_ms = sample_hmm(A_ms, pi_ms, ms_means, ms_stds, T_ms, seed=7)

# Decodificar con Viterbi
z_ms_hat = viterbi(X_ms, A_ms, pi_ms, ms_means, ms_stds)
acc_ms   = (z_ms_hat == z_ms).mean()

# Calcular estadísticas de duración
def estadisticas_duracion(z, fs=250):
    """Calcula duraciones en ms de cada run de estados."""
    duraciones = {k: [] for k in range(4)}
    t, n = 0, len(z)
    while t < n:
        k = z[t]; t2 = t
        while t2 < n and z[t2] == k: t2 += 1
        duraciones[k].append((t2 - t) * 1000 / fs)  # ms
        t = t2
    return duraciones

durs = estadisticas_duracion(z_ms)

fig, axes = plt.subplots(3, 1, figsize=(14, 8))

# Secuencia verdadera
t_axis = np.arange(T_ms) / 250 * 1000  # ms
for t in range(T_ms - 1):
    axes[0].fill_between([t_axis[t], t_axis[t+1]], [0, 0], [1, 1],
                          color=colores_ms[z_ms[t]], alpha=0.9, linewidth=0)
axes[0].set(yticks=[], xlim=[0, t_axis[-1]],
            title='Secuencia verdadera de microestados EEG')

# Secuencia Viterbi
for t in range(T_ms - 1):
    axes[1].fill_between([t_axis[t], t_axis[t+1]], [0, 0], [1, 1],
                          color=colores_ms[z_ms_hat[t]], alpha=0.9, linewidth=0)
axes[1].set(yticks=[], xlim=[0, t_axis[-1]],
            title=f'Viterbi decodificado  (exactitud={acc_ms:.3f})')

parches_ms = [mpatches.Patch(color=c, label=f'MS-{n}')
              for c, n in zip(colores_ms, nombres_ms)]
axes[0].legend(handles=parches_ms, loc='upper right', fontsize=8, ncol=4)

# Duraciones
axes[2].set_visible(True)
axes[2].set(title='Distribución de duración de microestados', xlabel='Duración (ms)')
for k in range(4):
    if durs[k]:
        axes[2].hist(durs[k], bins=25, alpha=0.5, color=colores_ms[k],
                      label=f'MS-{nombres_ms[k]} (media={np.mean(durs[k]):.0f} ms)',
                      density=True)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print('\nDuración media por microestado:')
for k in range(4):
    print(f'  MS-{nombres_ms[k]}: {np.mean(durs[k]):.1f} ms '
          f'(rango {np.min(durs[k]):.0f}–{np.max(durs[k]):.0f} ms)')

## Parte 6 — Extensiones del HMM y alternativas

| Modelo | Extensión | Uso biomédico |
|---|---|---|
| **HMM-GMM** | Emisiones de mezcla gaussiana | Modelos robustos de EEG/voz |
| **HMM izquierda-derecha** | A_{ij}=0 para j<i | Segmentación PQRST del ECG |
| **HMM factorial** | Múltiples cadenas independientes | Estados EMG + EEG simultáneos |
| **HMM jerárquico** | Estados dentro de estados | Ciclos del sueño + micro-despertares |
| **Deep HMM** | Emisiones con RNN/Transformer | Señales fisiológicas a gran escala |

In [ ]:
# ── Comparación: Viterbi vs posterior suavizada ───────────────────────────────
# Ilustra cuándo difieren: transiciones rápidas (Viterbi las suprime)

log_alpha_ms, _ = forward(X_ms, A_ms, pi_ms, ms_means, ms_stds)
log_beta_ms     = backward(X_ms, A_ms, ms_means, ms_stds)
log_gamma_ms    = log_alpha_ms + log_beta_ms
log_gamma_ms   -= logsumexp(log_gamma_ms, axis=1, keepdims=True)
gamma_ms        = np.exp(log_gamma_ms)
z_post_ms       = gamma_ms.argmax(axis=1)

acc_post_ms = (z_post_ms == z_ms).mean()
acc_vit_ms  = (z_ms_hat  == z_ms).mean()

# Segmento de 500 ms para visualización detallada
t0, t1 = 250, 375  # 500 ms
seg    = slice(t0, t1)

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)

# GFP
axes[0].plot(t_axis[seg], X_ms[seg, 0], 'k', lw=1)
axes[0].set(ylabel='GFP (μV)', title='GFP observada')

# Verdad
for t in range(t0, t1-1):
    axes[1].fill_between([t_axis[t], t_axis[t+1]], [0, 0], [1, 1],
                          color=colores_ms[z_ms[t]], alpha=0.9, linewidth=0)
axes[1].set(yticks=[], title='Estado verdadero')

# Viterbi
for t in range(t0, t1-1):
    axes[2].fill_between([t_axis[t], t_axis[t+1]], [0, 0], [1, 1],
                          color=colores_ms[z_ms_hat[t]], alpha=0.9, linewidth=0)
axes[2].set(yticks=[], title=f'Viterbi (acc={acc_vit_ms:.3f})')

# Posterior
for k in range(4):
    axes[3].plot(t_axis[seg], gamma_ms[seg, k], color=colores_ms[k],
                  lw=1.5, label=f'MS-{nombres_ms[k]}')
axes[3].set(ylabel='P(estado|X)', xlabel='Tiempo (ms)',
            title=f'Posterior suavizada (acc argmax={acc_post_ms:.3f})')
axes[3].legend(fontsize=8, ncol=4)

plt.suptitle('Comparación Viterbi vs posterior suavizada — microestados EEG\n'
              'Viterbi suprime transiciones rápidas; la posterior es más suave', y=1.01)
plt.tight_layout()
plt.show()

print(f'Exactitud Viterbi:   {acc_vit_ms:.3f}')
print(f'Exactitud posterior: {acc_post_ms:.3f}')

## ✏️ Ejercicios

Los ejercicios usan los datasets **ISRUC-Sleep** y **PhysioNet Sleep-EDF**.

1. **Suavizado vs filtrado.** El algoritmo Forward da el posterior **filtrado** P(zₜ|x₁,...,xₜ). El Forward-Backward da el posterior **suavizado** P(zₜ|X). Calcula ambos sobre la secuencia X_seq. ¿Cuál tiene mayor exactitud? ¿En qué tipo de transiciones difieren más? Visualiza la diferencia en una ventana de 50 épocas.

2. **Sensibilidad al número de estados.** Ajusta HMMs con K ∈ {3, 4, 5, 6} sobre los datos de sueño usando Baum-Welch. Selecciona K con BIC. ¿Coincide con los 5 estados del manual AASM? ¿Qué estados se unen cuando K=3?

3. **HMM izquierda-derecha para segmentación de ECG.** Para segmentar los complejos PQRST del ECG (cada segmento ocurre solo una vez por ciclo), modifica la matriz de transición para que A_{ij}=0 cuando j<i (no se puede retroceder). Genera una secuencia de 5 ciclos cardíacos simulados y decodifica con Viterbi.

4. **Efecto de los valores iniciales en Baum-Welch.** Ajusta 10 HMMs con semillas distintas sobre los datos de sueño. Grafica la distribución de log-verosimilitudes finales. ¿Qué fracción converge a la misma solución? ¿Cuántas iteraciones necesita cada uno?

5. *(Desafío)* **ISRUC-Sleep real.** Descarga el dataset ISRUC-Sleep (https://sleeptight.isr.uc.pt/). Extrae características espectrales (potencias por banda). Ajusta un HMM con K=5 via Baum-Welch. Evalúa la coincidencia con las etiquetas del experto usando κ de Cohen. Compara con un clasificador supervisado (QDA, Random Forest) sobre los mismos datos.

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| ISRUC-Sleep | Khalighi, S. et al. (2016). *Computers Methods Programs Biomed.* 124:180–192. https://sleeptight.isr.uc.pt/ | Dataset principal S11 — estadificación sueño |
| PhysioNet Sleep-EDF | Cassette, P. & Guilleminault, C. (1992). PhysioNet. https://physionet.org/content/sleep-edfx/ | Alternativa para Baum-Welch |
| MIT-BIH Waveform Database | Moody, G.B. & Mark, R.G. (2001). *IEEE Eng. Med. Biol.* 20(3):45–50. https://physionet.org/content/mitdb/ | Ejercicio HMM izquierda-derecha ECG |